<a href="https://colab.research.google.com/github/Doran1690/llm-engineering-projects/blob/main/gym_churn_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from huggingface_hub import login

In [4]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [6]:
MODEL = "meta-llama/Llama-3.2-3B-Instruct"

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", quantization_config=quant_config)

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [9]:
messages = [
    {"role":"system","content":"You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
    {"role":"user","content":"  Give me 5 customer reviews about a restaurant named Sicily Pizza."}
]

In [10]:
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=500)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


1. "Hands down the best pizza I've had in years! The Sicilian-style crust was perfect, and the toppings were fresh and flavorful. 10/10 would recommend!" - Emily W.

2. "I was a bit skeptical about trying a new pizza place, but Sicily Pizza did not disappoint. The atmosphere was cozy and intimate, and the service was top-notch. Highly recommend!" - David K.

3. "I'm a huge fan of garlic knots, and Sicily Pizza did not disappoint. They were crispy on the outside and soft on the inside, just like I like them. 5 stars!" - Rachel T.

4. "I ordered the veggie pizza and was a bit disappointed that it didn't have any meat options. However, the veggies were all fresh and the sauce was delicious. Would order again." - James B.

5. "Sicily Pizza has the best cannoli in town! The crust was crispy and the filling was creamy and sweet. A must-try dessert!" - Maria G.


In [11]:
def generate_data(request):
    messages = [
        {"role": "system", "content": "You are a helpful assistant who generates realistic synthetic data. The data should look real and varied. Reply only with the data itself, without any explanations."},
        {"role": "user", "content": request}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=500)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response

In [12]:
print(generate_data("Give me 5 fake customers with name, age and city."))

1. Emily Wilson - 32 - New York
2. David Kim - 41 - Los Angeles
3. Sophia Patel - 28 - Chicago
4. Michael Brown - 55 - Houston
5. Jessica Martin - 19 - Miami


In [13]:
import gradio as gr

gr.Interface(fn=generate_data, inputs="text", outputs="text").launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://838ad68d0d9d9d19b9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
csv_text = """
member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
3,2025-01-01,28,Holon,monthly,250,12,no,yes
4,2024-09-10,25,Ramat Gan,student,150,6,no,no
5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
7,2025-02-25,35,Holon,student,150,14,no,no
8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes
"""
print(csv_text)


member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
3,2025-01-01,28,Holon,monthly,250,12,no,yes
4,2024-09-10,25,Ramat Gan,student,150,6,no,no
5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
7,2025-02-25,35,Holon,student,150,14,no,no
8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes



In [26]:
import pandas as pd
from io import StringIO

df = pd.read_csv(StringIO(csv_text))
df

,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
0,1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,no
1,2,2024-03-22,41,Ramat Gan,annual,180,5,no,yes
2,3,2025-01-01,28,Holon,monthly,250,12,no,yes
3,4,2024-09-10,25,Ramat Gan,student,150,6,no,no
4,5,2025-05-20,41,Ramat Gan,annual,180,10,yes,yes
5,6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,no
6,7,2025-02-25,35,Holon,student,150,14,no,no
7,8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,yes
8,9,2024-04-15,28,Tel Aviv,annual,180,7,yes,no
9,10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,yes


In [27]:
df["join_date"] = pd.to_datetime(df["join_date"])

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   member_id         10 non-null     int64         
 1   join_date         10 non-null     datetime64[ns]
 2   age               10 non-null     int64         
 3   branch            10 non-null     object        
 4   plan              10 non-null     object        
 5   monthly_fee       10 non-null     int64         
 6   visits_per_month  10 non-null     int64         
 7   personal_trainer  10 non-null     object        
 8   churned           10 non-null     object        
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 852.0+ bytes


In [22]:
df["branch"].value_counts()

,count
branch,
Ramat Gan,5
Tel Aviv,3
Holon,2


In [23]:
df["plan"].value_counts()

,count
plan,
monthly,5
annual,3
student,2


In [28]:
df["churned"] = df["churned"].map({"yes": 1, "no": 0})
df

,member_id,join_date,age,branch,plan,monthly_fee,visits_per_month,personal_trainer,churned
0,1,2024-06-15,32,Tel Aviv,monthly,250,15,yes,0
1,2,2024-03-22,41,Ramat Gan,annual,180,5,no,1
2,3,2025-01-01,28,Holon,monthly,250,12,no,1
3,4,2024-09-10,25,Ramat Gan,student,150,6,no,0
4,5,2025-05-20,41,Ramat Gan,annual,180,10,yes,1
5,6,2024-11-18,29,Tel Aviv,monthly,250,18,yes,0
6,7,2025-02-25,35,Holon,student,150,14,no,0
7,8,2024-08-05,24,Ramat Gan,monthly,250,11,yes,1
8,9,2024-04-15,28,Tel Aviv,annual,180,7,yes,0
9,10,2024-10-22,43,Ramat Gan,monthly,250,19,yes,1


In [29]:
df["churned"].mean()

np.float64(0.5)

In [30]:
df.groupby("branch")["churned"].mean()

,churned
branch,
Holon,0.5
Ramat Gan,0.8
Tel Aviv,0.0
